# Part 3: NLP and Sequence Modeling Mini Project
## Customer Support Message Sentiment Classification

**Dataset:** Customer Support Text Classification Dataset  
**Target:** `sentiment_label` — positive, neutral, negative  
**Input:** `customer_message` — free-text customer support tickets  
**Objective:** Build an NLP pipeline that classifies customer messages by sentiment, compare traditional vectorization approaches against sequence-aware architectures, and reflect on the evolution from RNNs to transformers.


## Task 1: Text Dataset Understanding and Exploration

In [ ]:
# ---------- imports ----------
import re
import string
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
warnings.filterwarnings('ignore')

from collections import Counter

# NLP / ML
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (classification_report, confusion_matrix,
                              ConfusionMatrixDisplay, accuracy_score)
from sklearn.pipeline import Pipeline

# Deep learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

# download NLTK resources
for resource in ['punkt', 'stopwords', 'wordnet', 'averaged_perceptron_tagger',
                 'punkt_tab']:
    nltk.download(resource, quiet=True)

np.random.seed(42)
tf.random.set_seed(42)

print(f'TensorFlow : {tf.__version__}')
print(f'NLTK       : {nltk.__version__}')
print(f'Scikit-learn version loaded')

In [ ]:
# ---------- load dataset ----------
DATA_FILE = 'customer_support_text_classification.csv'
df = pd.read_csv(DATA_FILE)

print('=== Basic Dataset Info ===')
print(f'Shape            : {df.shape}  ({df.shape[0]} rows, {df.shape[1]} columns)')
print(f'Columns          : {list(df.columns)}')
print()
print('=== Data Types ===')
print(df.dtypes)
print()
print('=== Missing Values ===')
print(df.isnull().sum())
print()
print('=== Sentiment Distribution ===')
print(df['sentiment_label'].value_counts())
print()
print('=== Sample Records ===')
df[['customer_message', 'sentiment_label', 'word_count', 'channel']].head(6)

In [ ]:
# ---------- exploratory visualisations ----------
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('Dataset Exploration — Customer Support Sentiment', fontsize=14, fontweight='bold')

# 1. Sentiment distribution
ax = axes[0, 0]
counts = df['sentiment_label'].value_counts()
colors = {'positive': '#2ecc71', 'neutral': '#3498db', 'negative': '#e74c3c'}
bars = ax.bar(counts.index, counts.values,
              color=[colors[c] for c in counts.index], edgecolor='black', lw=0.8)
for bar, v in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 4, str(v),
            ha='center', fontweight='bold')
ax.set_title('Sentiment Label Distribution')
ax.set_xlabel('Sentiment'); ax.set_ylabel('Count')
ax.set_ylim(0, counts.max() + 50)

# 2. Word count distribution by sentiment
ax = axes[0, 1]
for label, col in colors.items():
    subset = df[df['sentiment_label'] == label]['word_count']
    ax.hist(subset, bins=15, alpha=0.6, label=label, color=col, edgecolor='white')
ax.set_title('Word Count Distribution by Sentiment')
ax.set_xlabel('Word Count'); ax.set_ylabel('Frequency')
ax.legend()

# 3. Channel distribution
ax = axes[1, 0]
ch_counts = df['channel'].value_counts()
ax.barh(ch_counts.index, ch_counts.values, color='steelblue', edgecolor='black', lw=0.8)
for i, v in enumerate(ch_counts.values):
    ax.text(v + 2, i, str(v), va='center', fontweight='bold')
ax.set_title('Tickets by Channel')
ax.set_xlabel('Count')

# 4. Sentiment x Channel heatmap
ax = axes[1, 1]
crosstab = pd.crosstab(df['channel'], df['sentiment_label'])
sns.heatmap(crosstab, annot=True, fmt='d', cmap='YlOrRd', ax=ax, linewidths=0.5)
ax.set_title('Sentiment Distribution Across Channels')
ax.set_xlabel('Sentiment'); ax.set_ylabel('Channel')

plt.tight_layout()
plt.savefig('results/dataset_exploration.png', dpi=150, bbox_inches='tight')
plt.show()

print('Key observation: the dataset is nearly balanced across all three sentiment classes.')
print('Social channel shows a slightly higher proportion of negative messages — typical of')
print('public platforms where frustrated customers are more likely to post.')

In [ ]:
# ---------- statistical summary ----------
print('=== Word Count Statistics ===')
print(df.groupby('sentiment_label')['word_count'].describe().round(2))
print()
print('=== Message Length (characters) ===')
df['msg_length'] = df['customer_message'].str.len()
print(df.groupby('sentiment_label')['msg_length'].describe().round(2))

## Task 2: Text Preprocessing and Cleaning

In [ ]:
# ---------- preprocessing pipeline ----------
STOPWORDS = set(stopwords.words('english'))
# retain sentiment-critical negation words
STOPWORDS -= {'not', 'no', 'never', 'nor', "isn't", "wasn't", "wouldn't",
              "couldn't", "shouldn't", "won't", "can't", "don't", "didn't"}

lemmatizer = WordNetLemmatizer()

def clean_text(text: str) -> str:
    """
    Full preprocessing pipeline:
      1. Lowercase
      2. Remove ticket numbers (e.g. 'ticket number is 12345')
      3. Remove punctuation and digits
      4. Tokenize
      5. Remove stopwords (keeping negations)
      6. Lemmatize
      7. Drop very short tokens
    """
    text = text.lower()
    # remove ticket numbers — they carry no sentiment signal
    text = re.sub(r'(ticket\s+number\s+is\s+\d+|my\s+ticket\s+is\s+\d+)', '', text)
    text = re.sub(r'\d+', '', text)                     # remaining digits
    text = text.translate(str.maketrans('', '', string.punctuation))  # punctuation
    text = re.sub(r'\s+', ' ', text).strip()            # extra whitespace
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in STOPWORDS]  # stopword removal
    tokens = [lemmatizer.lemmatize(t) for t in tokens]  # lemmatization
    tokens = [t for t in tokens if len(t) > 1]          # drop single chars
    return ' '.join(tokens)

# apply
df['clean_message'] = df['customer_message'].apply(clean_text)

print('=== Preprocessing Examples ===')
for _, row in df[['customer_message', 'clean_message', 'sentiment_label']].sample(4, random_state=7).iterrows():
    print(f'  Original : {row["customer_message"]}')
    print(f'  Cleaned  : {row["clean_message"]}')
    print(f'  Label    : {row["sentiment_label"]}')
    print()

In [ ]:
# ---------- most frequent words per class ----------
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Top 15 Words After Cleaning — Per Sentiment Class', fontsize=13, fontweight='bold')

label_colors = {'positive': '#27ae60', 'neutral': '#2980b9', 'negative': '#c0392b'}

for ax, label in zip(axes, ['positive', 'neutral', 'negative']):
    all_words = ' '.join(df[df['sentiment_label'] == label]['clean_message']).split()
    top = Counter(all_words).most_common(15)
    words, freqs = zip(*top)
    ax.barh(list(words)[::-1], list(freqs)[::-1],
            color=label_colors[label], edgecolor='black', lw=0.5)
    ax.set_title(label.capitalize(), fontsize=12, fontweight='bold', color=label_colors[label])
    ax.set_xlabel('Frequency')

plt.tight_layout()
plt.savefig('results/top_words_per_class.png', dpi=150, bbox_inches='tight')
plt.show()

## Task 3: Text Vectorization — Approach and Explanation

### Vectorization Approaches Explored

Raw text cannot be fed directly into machine learning models — it must be converted to numerical representations. Three progressively sophisticated approaches are used here:

---

**1. Bag of Words (BoW)**  
Each message is represented as a vector of word counts over the entire vocabulary. The position and order of words is ignored completely — only the presence and frequency of each word matters. A vocabulary built from 5,000 training tokens produces a sparse 5,000-dimensional vector for every message. BoW is simple, fast, and surprisingly effective for short texts where word choice matters more than word order.

*Limitation:* "The product is not good" and "The product is good" would produce nearly identical BoW vectors (differing only by the presence of "not"), yet they have opposite sentiments. BoW cannot capture negation, idioms, or context.

---

**2. TF-IDF (Term Frequency – Inverse Document Frequency)**  
TF-IDF scales each word count by how *rare* the word is across all documents. A word that appears in almost every message (e.g. "customer", "please") receives a low weight because it is not discriminative. A word that appears in only a handful of messages (e.g. "delighted", "appalled") receives a high weight because it is distinctive. This tends to emphasise sentiment-carrying content words and downweight generic filler.

*Formula:* `TF-IDF(t, d) = TF(t, d) × log(N / df(t))`  
where N is total documents and df(t) is the number of documents containing term t.

*Advantage over BoW:* Better signal-to-noise ratio — the model is directed toward words that actually discriminate classes.

---

**3. Integer Sequences (for deep learning)**  
For LSTM/RNN models, each message is converted to a sequence of integer token IDs (based on a fixed vocabulary), then padded or truncated to a fixed length. Unlike BoW/TF-IDF, the sequence structure is preserved — word order is maintained. The embedding layer then maps each integer to a dense, learned vector in a lower-dimensional space, allowing the model to discover that "not good" and "terrible" should have similar representations.


In [ ]:
# ---------- train / val / test split ----------
# split before fitting any vectorizer to prevent data leakage
LABEL_MAP = {'negative': 0, 'neutral': 1, 'positive': 2}
df['label_int'] = df['sentiment_label'].map(LABEL_MAP)
CLASS_NAMES = ['negative', 'neutral', 'positive']

X = df['clean_message']
y = df['label_int']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

print(f'Training   : {len(X_train)} samples')
print(f'Validation : {len(X_val)} samples')
print(f'Test       : {len(X_test)} samples')
print()
print('Training class distribution:')
print(pd.Series(y_train).map({v:k for k,v in LABEL_MAP.items()}).value_counts())

In [ ]:
# ---------- BoW and TF-IDF vectorizers ----------
bow_vec = CountVectorizer(max_features=5000, ngram_range=(1, 2))
tfidf_vec = TfidfVectorizer(max_features=5000, ngram_range=(1, 2),
                             sublinear_tf=True)

# fit ONLY on training data
X_train_bow   = bow_vec.fit_transform(X_train)
X_val_bow     = bow_vec.transform(X_val)
X_test_bow    = bow_vec.transform(X_test)

X_train_tfidf = tfidf_vec.fit_transform(X_train)
X_val_tfidf   = tfidf_vec.transform(X_val)
X_test_tfidf  = tfidf_vec.transform(X_test)

print(f'BoW matrix shape   (train) : {X_train_bow.shape}')
print(f'TF-IDF matrix shape(train) : {X_train_tfidf.shape}')
print(f'Sparsity of BoW matrix     : {1 - X_train_bow.nnz / np.prod(X_train_bow.shape):.4f}')

## Task 4: Baseline Models and Evaluation

In [ ]:
# ---------- helper: evaluate and report ----------
def evaluate_model(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    print(f'\n=== {name} ===')
    print(f'Test Accuracy: {acc:.4f}  ({acc*100:.2f}%)')
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))
    return acc

results = {}

In [ ]:
# ---------- Baseline 1: Naive Bayes + BoW ----------
nb_bow = MultinomialNB(alpha=0.5)
nb_bow.fit(X_train_bow, y_train)
y_pred_nb_bow = nb_bow.predict(X_test_bow)
results['NB + BoW'] = evaluate_model('Naive Bayes + Bag of Words', y_test, y_pred_nb_bow)

In [ ]:
# ---------- Baseline 2: Logistic Regression + TF-IDF ----------
lr_tfidf = LogisticRegression(C=1.0, max_iter=1000, random_state=42, solver='lbfgs',
                               multi_class='multinomial')
lr_tfidf.fit(X_train_tfidf, y_train)
y_pred_lr = lr_tfidf.predict(X_test_tfidf)
results['LR + TF-IDF'] = evaluate_model('Logistic Regression + TF-IDF', y_test, y_pred_lr)

In [ ]:
# ---------- confusion matrices side by side ----------
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Confusion Matrices — Baseline Models', fontsize=13, fontweight='bold')

for ax, y_pred, title in zip(
    axes,
    [y_pred_nb_bow, y_pred_lr],
    ['Naive Bayes + BoW', 'Logistic Regression + TF-IDF']
):
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(title, fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('results/confusion_matrices_baseline.png', dpi=150, bbox_inches='tight')
plt.show()

## Task 5: Sequence Model — Embedding + LSTM

In [ ]:
# ---------- tokenize for deep learning ----------
VOCAB_SIZE  = 5000
MAX_LEN     = 30     # 95th percentile of word count is ~20; 30 gives comfortable headroom
EMBED_DIM   = 64
NUM_CLASSES = 3

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)   # fit ONLY on training data

def texts_to_padded(texts):
    seqs = tokenizer.texts_to_sequences(texts)
    return pad_sequences(seqs, maxlen=MAX_LEN, padding='post', truncating='post')

X_train_seq = texts_to_padded(X_train)
X_val_seq   = texts_to_padded(X_val)
X_test_seq  = texts_to_padded(X_test)

y_train_arr = np.array(y_train)
y_val_arr   = np.array(y_val)
y_test_arr  = np.array(y_test)

print(f'Vocabulary size (top words) : {VOCAB_SIZE}')
print(f'Max sequence length         : {MAX_LEN}')
print(f'X_train_seq shape           : {X_train_seq.shape}')
print()
# show a tokenized example
example_text = X_train.iloc[0]
print(f'Example cleaned message: "{example_text}"')
print(f'As integer sequence    : {X_train_seq[0][:15]} ...')

In [ ]:
# ---------- build LSTM model ----------
# Architecture:
#   Embedding layer   - converts token IDs to dense 64-dim vectors
#   Bidirectional LSTM - reads the sequence in both directions to capture
#                        forward and backward context (e.g. "not good" vs "good not")
#   Dropout            - regularisation
#   Dense(64)          - intermediate representation
#   Dense(3, softmax)  - class probabilities

def build_lstm_model():
    model = keras.Sequential([
        layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM,
                         input_length=MAX_LEN, name='embedding'),
        layers.SpatialDropout1D(0.20),
        layers.Bidirectional(
            layers.LSTM(64, dropout=0.30, recurrent_dropout=0.20),
            name='bidirectional_lstm'
        ),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.30),
        layers.Dense(NUM_CLASSES, activation='softmax', name='output')
    ], name='sentiment_lstm')

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

lstm_model = build_lstm_model()
lstm_model.summary()

In [ ]:
# ---------- train LSTM ----------
early_stop = EarlyStopping(monitor='val_loss', patience=5,
                            restore_best_weights=True, verbose=1)

lstm_history = lstm_model.fit(
    X_train_seq, y_train_arr,
    validation_data=(X_val_seq, y_val_arr),
    epochs=30,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# ---------- evaluate LSTM ----------
lstm_loss, lstm_acc = lstm_model.evaluate(X_test_seq, y_test_arr, verbose=0)
print(f'LSTM Test Loss     : {lstm_loss:.4f}')
print(f'LSTM Test Accuracy : {lstm_acc:.4f}  ({lstm_acc*100:.2f}%)')

y_pred_lstm = np.argmax(lstm_model.predict(X_test_seq, verbose=0), axis=1)
results['Bi-LSTM'] = lstm_acc
print()
print(classification_report(y_test_arr, y_pred_lstm, target_names=CLASS_NAMES))

In [ ]:
# ---------- LSTM training curves ----------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
epochs_ran = range(1, len(lstm_history.history['accuracy']) + 1)

ax1.plot(epochs_ran, lstm_history.history['accuracy'],     label='Train', color='steelblue', lw=2)
ax1.plot(epochs_ran, lstm_history.history['val_accuracy'], label='Validation', color='coral', lw=2, ls='--')
ax1.set_title('LSTM — Accuracy', fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(epochs_ran, lstm_history.history['loss'],     label='Train', color='steelblue', lw=2)
ax2.plot(epochs_ran, lstm_history.history['val_loss'], label='Validation', color='coral', lw=2, ls='--')
ax2.set_title('LSTM — Loss', fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
ax2.legend(); ax2.grid(alpha=0.3)

fig.suptitle('Bidirectional LSTM Training History — Sentiment Classification', fontsize=13)
plt.tight_layout()
plt.savefig('results/lstm_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ---------- LSTM confusion matrix ----------
cm_lstm = confusion_matrix(y_test_arr, y_pred_lstm)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm_lstm, display_labels=CLASS_NAMES).plot(ax=ax, cmap='Purples')
ax.set_title('LSTM — Confusion Matrix (Test Set)', fontweight='bold')
plt.tight_layout()
plt.savefig('results/confusion_matrix_lstm.png', dpi=150)
plt.show()

In [ ]:
# ---------- model comparison chart ----------
fig, ax = plt.subplots(figsize=(8, 5))
model_names = list(results.keys())
accuracies  = [results[k] * 100 for k in model_names]
bar_colors  = ['#3498db', '#2ecc71', '#9b59b6']

bars = ax.bar(model_names, accuracies, color=bar_colors, edgecolor='black', lw=0.8)
for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{acc:.2f}%', ha='center', fontweight='bold', fontsize=11)

ax.set_title('Model Comparison — Test Accuracy', fontsize=13, fontweight='bold')
ax.set_ylabel('Test Accuracy (%)')
ax.set_ylim(0, 105)
ax.axhline(y=33.3, color='red', ls='--', lw=1.2, label='Random baseline (33.3%)')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('results/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('=== Summary Table ===')
for name, acc in results.items():
    print(f'  {name:<25} : {acc*100:.2f}%')

## Task 5 (continued): Sequence Architecture Explanation

### Why Sequence Models?

The baseline models (Naive Bayes, Logistic Regression) treat each message as an unordered bag of words. The phrase "not satisfied" would be tokenized to `{not, satisfied}` — identical in structure to `{satisfied, not}`. For short, formulaic customer support messages this limitation is manageable, but as text grows more complex the loss of word-order information becomes costly.

A **Bidirectional LSTM** processes the token sequence in both the forward and backward direction, maintaining a hidden state that accumulates context as it reads. By the time the LSTM reaches the word "satisfied", it has already processed "not", and that negation is encoded in the hidden state. This allows the model to understand that "not satisfied" is negative and "very satisfied" is positive — a distinction BoW/TF-IDF cannot reliably make.

### Embedding Layer

Before the LSTM, an **Embedding layer** maps each integer token ID to a learned dense vector (64 dimensions here). Initially random, these vectors are updated during backpropagation so that semantically related words ("frustrated", "annoyed", "upset") end up close together in the embedding space. This is a compact, continuous representation that generalises far better than the sparse, high-dimensional BoW vectors.

### Bidirectional LSTM vs Unidirectional LSTM

A standard LSTM reads left-to-right and produces a single hidden state at the end. A **Bidirectional LSTM** runs two LSTMs in parallel — one reads forward, one reads backward — and concatenates their outputs. For sentiment classification this is useful because the sentiment-carrying word sometimes appears at the beginning ("Unfortunately, the delivery was late") and sometimes at the end ("The delivery was, unfortunately, late"). Reading in both directions ensures neither is missed.


## Task 6: Reflection — RNNs, LSTMs, Attention, and Transformers

### Recurrent Neural Networks (RNNs)

An RNN processes a sequence one token at a time. At each step it takes the current token's embedding and the previous hidden state, and produces a new hidden state. In theory this gives the network "memory" of everything it has read. In practice, vanilla RNNs suffer from the **vanishing gradient problem**: as gradients are backpropagated through many time steps, they shrink exponentially, making it almost impossible for the network to learn dependencies between words that are far apart in the sequence. A sentence like "The product I ordered last month, despite the initial excitement, was disappointing" has the sentiment word ("disappointing") 14 tokens away from the subject ("product") — a vanilla RNN would likely forget the subject by the time it reaches the sentiment word.

### Long Short-Term Memory Networks (LSTMs)

LSTMs solve the vanishing gradient problem through a **gating mechanism**. Each LSTM cell has three gates:

| Gate | Function |
|---|---|
| **Forget gate** | Decides what fraction of the current cell state to discard |
| **Input gate** | Decides what new information to write into the cell state |
| **Output gate** | Decides what part of the cell state to expose as the hidden state |

The cell state acts as a "conveyor belt" that can carry information across many time steps with only multiplicative (not additive) interactions, so gradients can flow through it without vanishing. This allows LSTMs to capture long-range dependencies that standard RNNs cannot.

### Attention Mechanism

Even LSTMs have a bottleneck: the entire input sequence must be compressed into a single fixed-size hidden state vector before being passed to the classifier. For longer inputs this compression is lossy. The **attention mechanism** addresses this by allowing the model to "look back" at all hidden states rather than only the last one. At each output step, it computes a weighted sum of all encoder hidden states — the weights (attention scores) indicate how relevant each input token is to the current prediction. Intuitively, when classifying the sentiment of a customer message, the model can learn to attend heavily to words like "delighted" or "frustrated" and pay less attention to "my ticket number is" or "please respond".

### Transformers

Transformers (introduced in *Attention Is All You Need*, Vaswani et al., 2017) discard recurrence entirely and rely solely on **self-attention**. Every token in the input sequence attends to every other token simultaneously — there is no step-by-step reading. This has two major consequences:

1. **Parallelism:** All attention computations happen in parallel, making transformers dramatically faster to train on modern GPUs compared to sequential LSTMs.
2. **Long-range dependencies:** The distance between two tokens is irrelevant — the attention weight between them is computed directly, not accumulated through many recurrent steps.

Pre-trained transformer models like **BERT** (Bidirectional Encoder Representations from Transformers) are fine-tuned on downstream tasks like sentiment classification. A BERT model pre-trained on hundreds of millions of sentences already "understands" language at a deep level; fine-tuning it on 1,500 support tickets would likely push accuracy well above what any LSTM trained from scratch can achieve, because the model transfers knowledge from its massive pre-training corpus.

### Summary: Evolution of Sequence Modeling

| Model | Key strength | Key weakness |
|---|---|---|
| RNN | Sequential memory | Vanishing gradients, short-range dependencies |
| LSTM | Long-range dependencies via gates | Sequential computation, single bottleneck vector |
| Attention | Flexible focus on any input token | Still built on top of RNN/LSTM |
| Transformer | Fully parallel, global context | Requires large pre-training data |
| BERT / GPT | Transfer learning, state-of-the-art | Computationally expensive, large memory |

For this dataset of short, formulaic customer support messages, the difference between LSTM and transformer performance is expected to be small. On longer, more nuanced text (product reviews, legal documents, clinical notes), the transformer's advantage in capturing global context would be far more pronounced.


In [ ]:
# ---------- save LSTM model ----------
lstm_model.save('results/sentiment_lstm_model.keras')
print('LSTM model saved to results/sentiment_lstm_model.keras')

In [ ]:
# ---------- sample predictions ----------
print('=== Sample Predictions on Test Messages ===')
IDX_TO_LABEL = {v: k for k, v in LABEL_MAP.items()}
sample_indices = np.random.choice(len(X_test), 8, replace=False)
X_test_list = list(X_test)
orig_msgs   = list(df.loc[X_test.index, 'customer_message'])

for i in sample_indices:
    true_label = IDX_TO_LABEL[int(y_test_arr[i])]
    pred_label = IDX_TO_LABEL[int(y_pred_lstm[i])]
    correct    = '✓' if true_label == pred_label else '✗'
    print(f'{correct} Message : {orig_msgs[i][:90]}')
    print(f'  True: {true_label:8s}  |  Pred: {pred_label}')
    print()

In [ ]:
# ---------- final summary ----------
print('=' * 55)
print('            FINAL RESULTS SUMMARY')
print('=' * 55)
print(f'  Dataset              : {len(df)} customer messages')
print(f'  Classes              : {list(LABEL_MAP.keys())}')
print(f'  Training samples     : {len(X_train)}')
print(f'  Test samples         : {len(X_test)}')
print()
print('  Model Performances on Test Set:')
for name, acc in results.items():
    print(f'    {name:<28} : {acc*100:.2f}%')
print()
best = max(results, key=results.get)
print(f'  Best Model : {best}  ({results[best]*100:.2f}%)')
print('=' * 55)